In [5]:
# imports
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

In [7]:
# Noisy ring
rng = np.random.default_rng(0)
n = 60
R = 1.0
radial_sigma = 0.08

theta = rng.uniform(0, 2 * np.pi, size=n)
r = R + rng.normal(0.0, radial_sigma, size=n)
S = np.column_stack([r * np.cos(theta), r * np.sin(theta)])

# Parameters
max_edge_length = 0.9
n_frames = 140

# Pairwise distance matrix
D = np.linalg.norm(S[:, None, :] - S[None, :, :], axis=2)

# Edges
edges = []
for i, j in combinations(range(n), 2):
    f = D[i, j]
    if f <= max_edge_length:
        edges.append(((i, j), f))

# Triangles
triangles = []
for i, j, k in combinations(range(n), 3):
    f = max(D[i, j], D[i, k], D[j, k])
    if f <= max_edge_length:
        triangles.append(((i, j, k), f))

# Plot limits
pad = 0.2
xmin, ymin = S.min(axis=0) - pad
xmax, ymax = S.max(axis=0) + pad

fig, ax = plt.subplots(figsize=(6, 6))

def update(frame):
    eps = (frame / (n_frames - 1)) * max_edge_length
    ax.clear()

    for (i, j, k), f in triangles:
        if f <= eps:
            tri = S[[i, j, k]]
            ax.fill(tri[:, 0], tri[:, 1], color="C0", alpha=0.25)

    for (i, j), f in edges:
        if f <= eps:
            ax.plot(
                [S[i, 0], S[j, 0]],
                [S[i, 1], S[j, 1]],
                color="C0",
                lw=1
            )

    ax.scatter(S[:, 0], S[:, 1], color="black", s=20)
    ax.set_title("Vietoris-Rips complex (live)")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.text(0.02, 0.98, f"eps = {eps:.3f}", transform=ax.transAxes, va="top")

    return []

anim = FuncAnimation(fig, update, frames=n_frames, interval=40, blit=False)

html = HTML(anim.to_jshtml())
plt.close(fig)
display(html)